# Homework 4: FP-Growth
**Student Roll Number**: i231234  
**Course**: Data Mining / Frequent Pattern Mining


In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from itertools import combinations
from IPython.display import display

### 1. Transaction Construction

In [ ]:
df = pd.read_csv('campus_store_transactions.csv')

# Convert the 'Items' column into transaction lists
transactions = df['Items'].apply(lambda x: x.split('|')).tolist()
print("Total Transactions:", len(transactions))
print("First 3 transactions:")
print(transactions[:3])

### 2. Segment Based Baskets

In [ ]:
# Split transactions based on TimeOfDay
morning_transactions = df[df['TimeOfDay'] == 'Morning']['Items'].apply(lambda x: x.split('|')).tolist()
afternoon_transactions = df[df['TimeOfDay'] == 'Afternoon']['Items'].apply(lambda x: x.split('|')).tolist()
evening_transactions = df[df['TimeOfDay'] == 'Evening']['Items'].apply(lambda x: x.split('|')).tolist()
night_transactions = df[df['TimeOfDay'] == 'Night']['Items'].apply(lambda x: x.split('|')).tolist()

segments = {
    'Morning': morning_transactions,
    'Afternoon': afternoon_transactions,
    'Evening': evening_transactions,
    'Night': night_transactions
}

for seg, trans in segments.items():
    print(f"{seg} segments: {len(trans)}")

### 3. One Hot Encoding
Using `TransactionEncoder` from `mlxtend`.

In [ ]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

display(df_encoded.head())

### 4 & 5. FP-Tree Construction and FP-Growth Mining
Implementation of FP-Tree nodes, header tables, and recursive pattern mining from scratch.

In [ ]:
class TreeNode:
    def __init__(self, name, count, parent):
        self.name = name
        self.count = count
        self.nodeLink = None
        self.parent = parent
        self.children = {}
        
    def inc(self, numOccur):
        self.count += numOccur

def create_tree(dataset, min_sup_count):
    # Step 1: Compute Item Support
    header_table = {}
    for trans in dataset:
        for item in trans:
            header_table[item] = header_table.get(item, 0) + dataset[trans]
    
    # Step 2: Filter Infrequent Items based on min_sup_count
    for k in list(header_table.keys()):
        if header_table[k] < min_sup_count:
            del header_table[k]
            
    freq_item_set = set(header_table.keys())
    if len(freq_item_set) == 0:
        return None, None
        
    for k in header_table:
        header_table[k] = [header_table[k], None]
        
    # Step 4: Build the FP-Tree
    ret_tree = TreeNode('Null Set', 1, None)
    for tranSet, count in dataset.items():
        local_d = {}
        for item in tranSet:
            if item in freq_item_set:
                local_d[item] = header_table[item][0]
        if len(local_d) > 0:
            # Step 3: Sort Items by Global Frequency
            ordered_items = [v[0] for v in sorted(local_d.items(), key=lambda p: (-p[1], p[0]))]
            update_tree(ordered_items, ret_tree, header_table, count)
            
    return ret_tree, header_table

def update_tree(items, in_tree, header_table, count):
    if items[0] in in_tree.children:
        in_tree.children[items[0]].inc(count)
    else:
        in_tree.children[items[0]] = TreeNode(items[0], count, in_tree)
        if header_table[items[0]][1] is None:
            header_table[items[0]][1] = in_tree.children[items[0]]
        else:
            update_header(header_table[items[0]][1], in_tree.children[items[0]])
            
    if len(items) > 1:
        update_tree(items[1::], in_tree.children[items[0]], header_table, count)

def update_header(node_to_test, target_node):
    while node_to_test.nodeLink is not None:
        node_to_test = node_to_test.nodeLink
    node_to_test.nodeLink = target_node

def ascend_tree(leaf_node, prefix_path):
    if leaf_node.parent is not None:
        prefix_path.append(leaf_node.name)
        ascend_tree(leaf_node.parent, prefix_path)

def find_prefix_path(base_pat, tree_node):
    cond_pats = {}
    while tree_node is not None:
        prefix_path = []
        ascend_tree(tree_node, prefix_path)
        if len(prefix_path) > 1:
            cond_pats[frozenset(prefix_path[1:])] = tree_node.count
        tree_node = tree_node.nodeLink
    return cond_pats

def mine_tree(in_tree, header_table, min_sup_count, prefix, freq_item_list, total_trans):
    bigL = [v[0] for v in sorted(header_table.items(), key=lambda p: p[1][0])]
    for base_pat in bigL:
        new_freq_set = prefix.copy()
        new_freq_set.add(base_pat)
        support = header_table[base_pat][0] / total_trans
        freq_item_list[frozenset(new_freq_set)] = support
        
        cond_patt_bases = find_prefix_path(base_pat, header_table[base_pat][1])
        my_cond_tree, my_head = create_tree(cond_patt_bases, min_sup_count)
        
        if my_head is not None:
            mine_tree(my_cond_tree, my_head, min_sup_count, new_freq_set, freq_item_list, total_trans)

def fp_growth(transactions_list, min_sup=0.2):
    dataset = {}
    for trans in transactions_list:
        dataset[frozenset(trans)] = dataset.get(frozenset(trans), 0) + 1
        
    num_trans = len(transactions_list)
    min_sup_count = min_sup * num_trans
    
    tree, header_table = create_tree(dataset, min_sup_count)
    freq_items = {}
    if tree is not None:
        mine_tree(tree, header_table, min_sup_count, set(), freq_items, num_trans)
        
    return freq_items

print("FP-Tree and FP-Growth algorithm configured successfully.")

### 6. Association Rule Mining
Generating association rules from discovered frequent itemsets based on Confidence and Lift thresholds.

In [ ]:
def generate_rules(frequent_itemsets, min_conf=0.6, min_lift=1.0):
    rules = []
    for itemset in frequent_itemsets:
        if len(itemset) > 1:
            subsets = []
            for i in range(1, len(itemset)):
                subsets.extend([frozenset(s) for s in combinations(itemset, i)])
            
            for antecedent in subsets:
                consequent = itemset - antecedent
                
                sup_A = frequent_itemsets.get(antecedent, 0)
                sup_C = frequent_itemsets.get(consequent, 0)
                sup_AC = frequent_itemsets[itemset]
                
                if sup_A > 0 and sup_C > 0:
                    conf = sup_AC / sup_A
                    lift = conf / sup_C
                    
                    if conf >= min_conf and lift >= min_lift:
                        rules.append({
                            'antecedent': set(antecedent),
                            'consequent': set(consequent),
                            'support': sup_AC,
                            'confidence': conf,
                            'lift': lift
                        })
    return rules

print("Rule Generation algorithm configured successfully.")

### 7. Pattern Analysis and Summary
Running the mining processes separately to compare all four segments based on min_sup = 0.2, min_conf = 0.6, and min_lift = 1.0.

In [ ]:
summary_table = []

for segment_name, segment_trans in segments.items():
    print(f"\n{'='*50}")
    print(f" Segment: {segment_name} ({len(segment_trans)} transactions)")
    print(f"{'='*50}")
    
    # Mine Frequent Itemsets
    freq_itemsets = fp_growth(segment_trans, min_sup=0.2)
    
    num_freq = len(freq_itemsets)
    if num_freq > 0:
        max_len = max(len(itemset) for itemset in freq_itemsets.keys())
        top_5_freq = sorted(freq_itemsets.items(), key=lambda x: x[1], reverse=True)[:5]
    else:
        max_len = 0
        top_5_freq = []
        
    print(f"Number of frequent itemsets: {num_freq}")
    print(f"Largest itemset length: {max_len}")
    print("Top 5 frequent itemsets (by support):")
    for itemset, sup in top_5_freq:
        print(f"  {set(itemset)}: {sup:.2f}")
        
    # Generate Association Rules
    rules = generate_rules(freq_itemsets, min_conf=0.6, min_lift=1)
    rules = sorted(rules, key=lambda x: x['lift'], reverse=True)
    
    print(f"\nNumber of valid rules: {len(rules)}")
    print("Top 5 rules ranked by Lift:")
    for r in rules[:5]:
        print(f"  {list(r['antecedent'])} -> {list(r['consequent'])} | Support: {r['support']:.2f}, Conf: {r['confidence']:.2f}, Lift: {r['lift']:.2f}")
        
    summary_table.append({
        'Segment': segment_name,
        'Number of Transactions': len(segment_trans),
        'Frequent Itemsets (sup >= 0.2)': num_freq,
        'Largest Itemset Length': max_len,
        'Valid Rules (Conf>=0.6, Lift>=1)': len(rules)
    })

print("\n\n" + "*"*60)
print("                      SUMMARY TABLE                       ")
print("*"*60)
display(pd.DataFrame(summary_table))